<a href="https://colab.research.google.com/github/amilynestes1028/ds2002-fa26/blob/main/notebooks/03-pandas/Copy_of_2026_09_23_%E2%80%94_Cleaning_Clinic_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
# TODO
import numpy as np
import pandas as pd

# Inspect initial dataset properties
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nNull Count per Column:\n", df.isnull().sum())
print("\nExact Duplicate Rows:", df.duplicated().sum())

Shape: (8, 6)

Data Types:
 order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

Null Count per Column:
 order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

Exact Duplicate Rows: 1


**What is wrong with this data?** List at least five specific problems:

1. Exact duplicates
2. inconsistent text
3. invalid values
4. inconsistent data formats
5. missing values

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
removed = df.duplicated().sum()
clean = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
# Clean price column: strip '$' and whitespace, then cast to float
clean['price'] = clean['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)

# Verify the resulting data type is float
assert clean['price'].dtype == float

# Log the cleaning decision
log('price_cleaning', 'stripped dollar signs and whitespace, cast from text to float', len(clean))

[price_cleaning] stripped dollar signs and whitespace, cast from text to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [6]:
# Coerce quantity to numeric (converts 'NULL' string to NaN)
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

# Identify affected rows
missing_mask = clean['qty'].isna()
negative_mask = clean['qty'] < 0

missing_count = missing_mask.sum()
negative_count = negative_mask.sum()

# Decision 1: Fill missing quantity with median (1) assuming a typical transaction size of 1
clean['qty'] = clean['qty'].fillna(1)
log('qty_missing', 'imputed missing quantity with default/median value of 1', missing_count)

# Decision 2: Convert negative quantity to positive assuming an absolute value recording error
clean['qty'] = clean['qty'].abs().astype(int)
log('qty_negative', 'converted negative quantity (-3) to absolute value assuming data entry sign error', negative_count)

# Assert quantity is now integer type
assert np.issubdtype(clean['qty'].dtype, np.integer)

[qty_missing] imputed missing quantity with default/median value of 1 (1 row(s))
[qty_negative] converted negative quantity (-3) to absolute value assuming data entry sign error (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [7]:
print('before:', sorted(clean['category'].unique()))

start_categories = clean['category'].nunique()

# Lowercase, strip whitespace, and normalize punctuation
clean['category'] = (
    clean['category']
    .str.lower()
    .str.strip()
    .str.replace('-', '', regex=False)
)

# Explicit mapping to canonical category names
CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Apparel',
    'raingear': 'Rain Gear'
}

clean['category'] = clean['category'].map(CATEGORY_MAP).fillna(clean['category'])

end_categories = clean['category'].nunique()

print('after: ', sorted(clean['category'].unique()))

# Log the collapse decision
log('category_normalization', f'collapsed categories from {start_categories} to {end_categories} distinct values', len(clean))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Apparel', 'Food', 'Merch', 'Rain Gear']
[category_normalization] collapsed categories from 6 to 4 distinct values (7 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [8]:
# TODO
print('before:', clean['item'].to_list())

start_items = clean['item'].dropna().nunique()

# Clean strings: strip trailing/leading whitespace and standardize case
clean['item'] = clean['item'].str.strip().str.title()

# Map duplicate naming variants to a canonical item name
ITEM_MAP = {
    'Cheese Burger': 'Cheeseburger'
}
clean['item'] = clean['item'].replace(ITEM_MAP)

# Decision: Impute missing item name in Order 7 based on known price ($12.00) and category ('Merch')
missing_item_mask = clean['item'].isna()
missing_item_count = missing_item_mask.sum()

# Price $12.00 + Merch category uniquely corresponds to 'Foam Finger'
clean.loc[missing_item_mask, 'item'] = 'Foam Finger'

end_items = clean['item'].nunique()

print('after: ', clean['item'].to_list())

# Log cleaning and imputation decisions
log('item_normalization', f'standardized names and collapsed {start_items} variants into {end_items} distinct items', len(clean))
log('item_imputation', 'imputed missing item name in Order 7 as Foam Finger based on price ($12.00) and category (Merch)', missing_item_count)

before: ['Cheeseburger', 'cheese burger', 'Foam Finger', 'UVA T-Shirt ', 'Rain Poncho', 'rain poncho', nan]
after:  ['Cheeseburger', 'Cheeseburger', 'Foam Finger', 'Uva T-Shirt', 'Rain Poncho', 'Rain Poncho', 'Foam Finger']
[item_normalization] standardized names and collapsed 6 variants into 4 distinct items (7 row(s))
[item_imputation] imputed missing item name in Order 7 as Foam Finger based on price ($12.00) and category (Merch) (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [9]:
# TODO
# Parse timestamps into standardized datetime objects, coercing invalid/missing formats to NaT
clean['ts'] = pd.to_datetime(clean['ts'], format='mixed', errors='coerce')

# Count conversion failures / missing timestamps
nat_count = clean['ts'].isna().sum()

# Extract hour component into a new column (contains NaN where ts is NaT)
clean['hour'] = clean['ts'].dt.hour

# Log decisions
log('ts_parsing', f'parsed timestamps to datetime dtype with {nat_count} NaT failure(s)', len(clean))
log('feature_extraction', 'extracted hour column from parsed timestamps', clean['hour'].notna().sum())

# Verify column type
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])

print("Parsed timestamps:\n", clean[['ts', 'hour']])

[ts_parsing] parsed timestamps to datetime dtype with 1 NaT failure(s) (7 row(s))
[feature_extraction] extracted hour column from parsed timestamps (6 row(s))
Parsed timestamps:
                    ts  hour
0 2026-09-05 12:03:00  12.0
2 2026-09-05 12:40:00  12.0
3 2026-09-05 13:00:00  13.0
4 2026-09-05 13:05:00  13.0
5 2026-09-05 13:20:00  13.0
6                 NaT   NaN
7 2026-09-05 14:00:00  14.0


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [10]:
# 1. No exact duplicate rows remain
assert clean.duplicated().sum() == 0, "Duplicate rows detected"

# 2. Key numeric columns have correct types and non-negative values
assert pd.api.types.is_float_dtype(clean['price']), "Price column must be float"
assert pd.api.types.is_integer_dtype(clean['qty']), "Quantity column must be integer"
assert (clean['price'] >= 0).all(), "Negative prices found"
assert (clean['qty'] > 0).all(), "Non-positive quantities found"

# 3. No missing values in critical descriptive fields
assert clean['item'].isna().sum() == 0, "Missing item names detected"
assert clean['category'].isna().sum() == 0, "Missing categories detected"

# 4. Text standardization check (no trailing spaces or unmapped lowercase names)
assert clean['item'].str.startswith(' ').sum() == 0 and clean['item'].str.endswith(' ').sum() == 0, "Unstripped whitespace in item names"
assert set(clean['category'].unique()) == {'Food', 'Merch', 'Apparel', 'Rain Gear'}, "Unexpected category set"

# 5. Timestamp and derived feature validation
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), "Timestamp must be datetime dtype"
assert clean['hour'].dropna().between(0, 23).all(), "Invalid hour values extracted"


# --- Compute metrics ---
clean['revenue'] = clean['qty'] * clean['price']

total_rows = len(clean)
total_units = clean['qty'].sum()
total_revenue = clean['revenue'].sum()
distinct_categories = clean['category'].nunique()

print(f"Total Rows:            {total_rows}")
print(f"Total Units Sold:      {total_units}")
print(f"Total Revenue:         ${total_revenue:,.2f}")
print(f"Distinct Categories:   {distinct_categories}")

print("\nPipeline Execution Decisions:")
print(show_log().to_string(index=False))

Total Rows:            7
Total Units Sold:      14
Total Revenue:         $136.50
Distinct Categories:   4

Pipeline Execution Decisions:
                  step                                                                                         decision  rows
            duplicates                                                                     dropped exact duplicate rows     1
        price_cleaning                                    stripped dollar signs and whitespace, cast from text to float     7
           qty_missing                                          imputed missing quantity with default/median value of 1     1
          qty_negative                converted negative quantity (-3) to absolute value assuming data entry sign error     1
category_normalization                                                 collapsed categories from 6 to 4 distinct values     7
    item_normalization                                standardized names and collapsed 6 variants into 4 d

### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [11]:
print(show_log().to_string(index=False))

                  step                                                                                         decision  rows
            duplicates                                                                     dropped exact duplicate rows     1
        price_cleaning                                    stripped dollar signs and whitespace, cast from text to float     7
           qty_missing                                          imputed missing quantity with default/median value of 1     1
          qty_negative                converted negative quantity (-3) to absolute value assuming data entry sign error     1
category_normalization                                                 collapsed categories from 6 to 4 distinct values     7
    item_normalization                                standardized names and collapsed 6 variants into 4 distinct items     7
       item_imputation imputed missing item name in Order 7 as Foam Finger based on price ($12.00) and category (Merch

**The decision that mattered most:** Converting the negative quantity (-3) to a positive value (+3) assuming a keyboard entry sign error rather than an actual return/refund.

**Revenue with it:** 99
  **Revenue without it:** 63

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# Checkpoint
rows_after = len(clean)
revenue_after = clean['revenue'].sum()
biggest_decision = 'converting negative quantity (-3) to absolute value (+3)'
revenue_other_way = 63.0

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 7
revenue: 136.5
decision that mattered: converting negative quantity (-3) to absolute value (+3)
revenue the other way: 63.0
